# Prerequisite-event maxent recovery — mixture-base + estimator GPU sweep

An event `e` has `m` independent prerequisites `a_1..a_m`. Elicited
constraints are only `P(a_i) = 0.7` and the implications
`P(e ∧ ¬a_i) ≈ 0` — **nothing** about `P(e | all met)`. The true
max-entropy joint (star-shaped Gibbs family, see `maxent_reference`)
has `P(e | all met) = 0.5` exactly. Previously fitted flows collapsed
`P(e | all met)` far below 0.5: the implication constraints squash `e`
globally and the flow fails to carve the all-met corner back out.

Two questions, selected by `MODE` in the knobs cell:

* **`"ab"`** (default): which violation-gradient estimator recovers the
  corner best — sharp-**soft** sigmoids (pathwise/transport only),
  **st**raight-through indicators (exact forward, biased soft backward),
  or the **hybrid** pathwise+score estimator (exact forward, *unbiased*
  gradient via the `log q` score channel)? Each ± the weak-`k_hard`
  warmup that defuses the init-slam attractor.
* **`"full"`**: does a Gaussian-mixture base (`n_components=K`) help,
  across `m ∈ {2,4,6}` and `K ∈ {1,4,16,64}` + a flowless GMM arm.

**All experiment logic lives in `benchmarks/prereq_experiment.py`** (in the
repo). This notebook is a thin shell: the setup cell **clones-or-`git
pull`s** the repo, so iterating on the sweep is *edit
`prereq_experiment.py` → push → rerun the setup cell* — no notebook
re-upload. NOTE: `main` must carry the mixture-base and hybrid-estimator
commits; the import-guards below check.

**Runtime → Change runtime type → GPU** (T4 is fine), then Run all.

In [ ]:
# --- Setup: clone-or-pull the repo, install deps Colab lacks, keep Colab's jax ---
import os, sys, subprocess

REPO = "/content/calibrated_response"
BRANCH = "main"   # branch carrying the mixture-base + hybrid-estimator changes
URL = "https://github.com/amdson/calibrated_response.git"

if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, URL, REPO],
                   check=True)
else:
    # re-run picks up the latest payload without a re-clone (or notebook re-upload)
    subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", BRANCH],
                   check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", f"origin/{BRANCH}"],
                   check=True)
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

# optax/jaxopt are the only solver deps Colab doesn't ship; installing the
# package itself would drag in pinned jax/jaxlib and clobber the GPU build.
%pip -q install optax jaxopt

import jax
jax.config.update("jax_compilation_cache_dir", f"{REPO}/.jax_cache")
jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
print("jax backend:", jax.default_backend(), jax.devices())
assert jax.default_backend() != "cpu", "No GPU — switch the runtime type first"

# import-guard: confirm the pulled branch has the mixture base AND the
# hybrid (pathwise+score) estimator plumbing
import inspect
from calibrated_response.maxent_sampler.flow_model import FlowSamplerModel
assert "n_components" in inspect.signature(FlowSamplerModel).parameters, \
    "pulled branch predates the mixture-base commit — push it first"
assert "with_logq" in inspect.signature(FlowSamplerModel.constraint_loss).parameters, \
    "pulled branch predates the hybrid-estimator commit — push it first"
from benchmarks.prereq_experiment import (run_sweep, plot_results,
                                          default_configs,
                                          estimator_ab_configs, summarize_ab)
print(f"payload OK — {len(default_configs())} configs in the default grid, "
      f"{len(estimator_ab_configs())} in the estimator A/B")

In [ ]:
# --- Sweep knobs -----------------------------------------------------------
# MODE = "quick": tiny grid to validate the notebook end-to-end (~minutes).
# MODE = "ab":    the estimator comparison — soft vs ST vs hybrid violation
#                 scoring, each +/- weak-k warmup, m in {4, 6}, K=1.
#                 12 configs x 3 seeds; own results file (estimator_ab.jsonl).
# MODE = "full":  the full m x K grid (+ GMM arm) from default_configs().
MODE = "ab"

if MODE == "quick":
    CONFIGS = [
        dict(n_prereq=4, n_components=1, n_layers=6, hidden=64),
        dict(n_prereq=4, n_components=16, n_layers=6, hidden=64),
        dict(n_prereq=4, n_components=1, n_layers=6, hidden=64,
             warmup_steps=200, viol_mode="hybrid"),
    ]
    SEEDS, STEPS, N_SAMPLES, EVAL = (0,), 800, 2048, 50_000
    OUT = "results/prereq_quick.jsonl"
elif MODE == "ab":
    CONFIGS = estimator_ab_configs(ms=(4, 6))
    SEEDS, STEPS, N_SAMPLES, EVAL = (0, 1, 2), 2000, 4096, 200_000
    OUT = "results/estimator_ab.jsonl"
else:
    CONFIGS = default_configs()
    SEEDS, STEPS, N_SAMPLES, EVAL = (0, 1, 2), 3000, 4096, 200_000
    OUT = "results/prereq_sweep.jsonl"
print(f"{MODE}: {len(CONFIGS)} configs x {len(SEEDS)} seeds -> {OUT}")

In [ ]:
# --- Run the sweep (resumable: rows append to the OUT jsonl) ----------------
rows = run_sweep(configs=CONFIGS, seeds=SEEDS, steps=STEPS,
                 n_samples=N_SAMPLES, eval_samples=EVAL, out_path=OUT)
print(f"\n{len(rows)} rows total in {OUT}")

In [ ]:
# --- Summarize / plot -------------------------------------------------------
# "ab" mode: mean+-sd table per (m, viol_mode, warmup) arm — the direct
# soft-vs-ST-vs-hybrid readout.  Other modes: the K-axis recovery plot.
if MODE == "ab":
    summarize_ab(rows)
else:
    plot_results(rows, save="results/prereq_sweep.png");

In [ ]:
# --- Download results (unzip into results/ locally to merge) ---
import shutil
shutil.make_archive("/content/prereq_sweep_results", "zip", "results")
try:
    from google.colab import files
    files.download("/content/prereq_sweep_results.zip")
except ImportError:
    print("not on Colab — results in results/")